# PixelPlay word-sync model: download the training data once (CPU, no GPU hours)
Downloads and shrinks every dataset (MUSDB18-HQ real songs, their lyrics, children's songs, JamendoLyrics),
then packs them into `pp_data_pack/*.tar` in the Output. The training notebook reads them from there in
a minute instead of downloading for hours.

1. Right panel → **Session options**: Accelerator **None** (CPU only, so no GPU quota is used), Internet **On**.
2. **Save Version → Save & Run All (Commit)**.
3. If it times out (Zenodo can be very slow), open the notebook again, **Add Input → this notebook's own
   Output**, and run it again: it continues where it stopped.
4. When the log ends with `READY`, open the training notebook, **Add Input → this notebook's Output**, run it.

In [ ]:
DATA = "/kaggle/working/pp_stage"         # kept in the Output while downloading (resumable)
PACK = "/kaggle/working/pp_data_pack"
PREPARED_SOURCES = ["/kaggle/input"]      # a previous (timed-out) version of this notebook, attached as input
USE_GTSINGER = False
import os, subprocess
os.makedirs(DATA, exist_ok=True)
print(subprocess.run("df -h /kaggle/working", shell=True, capture_output=True, text=True).stdout)

In [ ]:
!pip -q install "soundfile>=0.12" musdb huggingface_hub remotezip

## Data: real songs first (downloads itself, 30-60 min)
Every source is optional: if one fails, it says why and the rest carry on.

| Source | What | Used for |
|---|---|---|
| **MUSDB18-HQ + lyrics extension** | 141 real, professionally produced songs (rock, pop, rap, electronic...), full mix *and* isolated vocals, lyrics in 3-12 s sections | training (real songs) |
| **Your library** (optional) | your own songs + synced `.lrc` lyrics | training (closest to what you listen to) |
| GTSinger English | studio singing with word labels | training |
| Children's Song Dataset | 50 sung songs + lyrics | training |
| MUSDB18 accompaniment | the instrumental part of each song (mix minus vocals) | mixed under clean vocals |
| **JamendoLyrics** | real songs with human word times | **evaluation only, never trained on** |

In [ ]:
import os, glob, zipfile, subprocess, traceback, math
import numpy as np, soundfile as sf
from scipy.signal import resample_poly
STATUS = {}
def step(name, fn):
    try:
        fn(); STATUS[name] = "ok"
    except Exception as e:
        traceback.print_exc(); STATUS[name] = f"FAILED: {e}"
    print(f"== {name}: {STATUS[name]}")

# 0. Data already prepared by kaggle_data_prep.ipynb (or a run that timed out)? Use it instead of
#    downloading again: *.tar packs and/or an unpacked pp_stage folder.
import tarfile, shutil
def restore_prepared():
    found = 0
    for root in PREPARED_SOURCES:
        for tar in sorted(glob.glob(f"{root}/**/pp_data_pack/*.tar", recursive=True)):
            name = os.path.basename(tar)[:-4]
            if os.path.exists(f"{DATA}/.restored_{name}"): continue
            print("restoring", tar)
            with tarfile.open(tar) as t: t.extractall(DATA)
            open(f"{DATA}/.restored_{name}", "w").close(); found += 1
        for stage in glob.glob(f"{root}/**/pp_stage", recursive=True):
            if os.path.abspath(stage) == os.path.abspath(DATA): continue
            print("restoring", stage)
            shutil.copytree(stage, DATA, dirs_exist_ok=True); found += 1
    print("prepared data restored from", found, "source(s)" if found else "- none attached, downloading")
restore_prepared()

def to16k_mono(x, sr):
    x = x.mean(axis=1) if x.ndim == 2 else x
    g = math.gcd(int(sr), 16000)
    return resample_poly(x, 16000 // g, int(sr) // g).astype(np.float32)

# 1. MUSDB18-HQ: fetch only mixture + vocals of each song straight out of the 22.7 GB zip
#    (HTTP range requests), shrink to 16 kHz mono, and derive the accompaniment as mix - vocals.
MUS = f"{DATA}/musdb16k"; ACC = f"{DATA}/accompaniment"
def get_musdb():
    # Zenodo sometimes drops a connection mid-file. Each song is retried (re-opening the zip)
    # and a song that keeps failing is skipped, so one bad moment no longer loses all 150.
    import time
    from remotezip import RemoteZip
    URL = "https://zenodo.org/records/3338373/files/musdb18hq.zip?download=1"
    os.makedirs(ACC, exist_ok=True)
    have = len(glob.glob(f"{MUS}/*/*/vocals.wav"))
    if have >= 140:
        print(f"  {have} songs already prepared - no download needed"); return
    def open_zip():
        for attempt in range(6):
            try:
                return RemoteZip(URL)
            except Exception as e:
                print(f"  opening the zip failed ({e}); retrying in {15 * (attempt + 1)} s")
                time.sleep(15 * (attempt + 1))
        raise RuntimeError("could not open the MUSDB18-HQ zip on Zenodo")
    z = open_zip()
    names = [i.filename for i in z.infolist() if i.filename.endswith("/mixture.wav")]
    done = {os.path.basename(os.path.dirname(p)) for p in glob.glob(f"{MUS}/*/*/vocals.wav")}
    failed = []
    for k, mix_name in enumerate(names):
        track_dir = os.path.dirname(mix_name); track = os.path.basename(track_dir)
        if track in done: continue
        split = "test" if "/test/" in f"/{mix_name}" else "train"
        out = f"{MUS}/{split}/{track}"
        tmp = f"{DATA}/_tmp"
        for attempt in range(5):
            try:
                subprocess.run(["rm", "-rf", tmp]); os.makedirs(tmp, exist_ok=True)
                z.extract(mix_name, tmp); z.extract(f"{track_dir}/vocals.wav", tmp)
                mix, sr = sf.read(f"{tmp}/{mix_name}", dtype="float32"); voc, _ = sf.read(f"{tmp}/{track_dir}/vocals.wav", dtype="float32")
                m16, v16 = to16k_mono(mix, sr), to16k_mono(voc, sr)
                os.makedirs(out, exist_ok=True)
                sf.write(f"{out}/mixture.wav", m16, 16000); sf.write(f"{out}/vocals.wav", v16, 16000)
                sf.write(f"{ACC}/{track}.wav", m16 - v16, 16000)
                print(f"  [{k+1}/{len(names)}] {track}")
                break
            except Exception as e:
                wait = 10 * (attempt + 1)
                print(f"  [{k+1}/{len(names)}] {track}: download broke ({str(e)[:80]}); retry {attempt+1}/5 in {wait} s")
                time.sleep(wait)
                try: z = open_zip()
                except Exception: pass
        else:
            failed.append(track)
            subprocess.run(["rm", "-rf", out])
        subprocess.run(["rm", "-rf", tmp])
    got = len(glob.glob(f"{MUS}/*/*/vocals.wav"))
    print(f"  songs ready: {got}/{len(names)}" + (f"; skipped after retries: {failed}" if failed else ""))
    if got < 50:
        raise RuntimeError(f"only {got} MUSDB songs downloaded - Zenodo is struggling, try again later")
step("MUSDB18-HQ audio", get_musdb)

def get_musdb_lyrics():
    os.makedirs(f"{DATA}/musdb_lyrics", exist_ok=True)
    if len(glob.glob(f"{DATA}/musdb_lyrics/**/*.txt", recursive=True)) >= 100:
        print("lyrics files:", len(glob.glob(f"{DATA}/musdb_lyrics/**/*.txt", recursive=True))); return
    for part in ("train_lyrics.zip", "test_lyrics.zip"):
        z = f"{DATA}/{part}"
        if not os.path.exists(z):
            subprocess.run(["wget", "-q", "-O", z, f"https://zenodo.org/records/3989267/files/{part}?download=1"], check=True)
        zipfile.ZipFile(z).extractall(f"{DATA}/musdb_lyrics")
    print("lyrics files:", len(glob.glob(f"{DATA}/musdb_lyrics/**/*.txt", recursive=True)))
step("MUSDB18 lyrics extension", get_musdb_lyrics)

# 2. GTSinger English (8.7 GB)
def get_gtsinger():
    # GTSinger is ~20k small files and Hugging Face allows 5000 file requests per 5 minutes, so
    # fetch only what training reads (audio + word labels), and on a rate limit wait it out and
    # carry on: files already downloaded are skipped on the next attempt.
    import time
    from huggingface_hub import snapshot_download
    for attempt in range(15):
        try:
            snapshot_download("GTSinger/GTSinger", repo_type="dataset",
                              allow_patterns=["English/**/*.wav", "English/**/*.json"],
                              local_dir=f"{DATA}/GTSinger", max_workers=4,
                              token=os.environ.get("HF_TOKEN") or None)
            break
        except Exception as e:
            if "429" not in str(e):
                raise
            print(f"  Hugging Face rate limit (attempt {attempt+1}); waiting 5 min, then resuming...")
            time.sleep(310)
    else:
        print("  still rate-limited; continuing with the GTSinger files downloaded so far")
    print("wavs:", len(glob.glob(f"{DATA}/GTSinger/English/**/*.wav", recursive=True)))
if USE_GTSINGER:
    step("GTSinger English", get_gtsinger)
else:
    print("== GTSinger English: skipped (USE_GTSINGER = False; run 1 spent 90 min on rate limits and got 0 usable clips)")

# 3. Children's Song Dataset (1.9 GB)
def get_csd():
    if not glob.glob(f"{DATA}/CSD/**/english", recursive=True):
        subprocess.run(["wget", "-q", "-O", f"{DATA}/CSD.zip", "https://zenodo.org/records/4785016/files/CSD.zip?download=1"], check=True)
        zipfile.ZipFile(f"{DATA}/CSD.zip").extractall(f"{DATA}/CSD"); os.remove(f"{DATA}/CSD.zip")
step("Children's Song Dataset", get_csd)

# 4. Fallback accompaniment if MUSDB18-HQ didn't work: the free 7 s previews
def get_previews():
    os.makedirs(ACC, exist_ok=True)
    if len(os.listdir(ACC)) < 20:
        import musdb
        for t in musdb.DB(root=f"{DATA}/musdb7", download=True):
            sf.write(f"{ACC}/{t.name.replace('/', '_')}_preview.wav", t.targets["accompaniment"].audio, t.rate)
    print("accompaniment files:", len(os.listdir(ACC)))
step("accompaniment", get_previews)

# 5. JamendoLyrics (evaluation only)
def get_jamendo():
    if not os.path.isdir(f"{DATA}/jamendolyrics"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/f90/jamendolyrics", f"{DATA}/jamendolyrics"], check=True)
step("JamendoLyrics", get_jamendo)

import json
print(json.dumps(STATUS, indent=1))
print(subprocess.run(f"du -sh {DATA}/* ; df -h {DATA}", shell=True, capture_output=True, text=True).stdout)

In [ ]:
# Pack into a few tar files (fast to attach and extract), then remove the unpacked copy.
import tarfile, os, glob, shutil, json
os.makedirs(PACK, exist_ok=True)
ok = len(glob.glob(f"{DATA}/musdb16k/*/*/vocals.wav")) >= 50 and len(glob.glob(f"{DATA}/musdb_lyrics/**/*.txt", recursive=True)) >= 100
parts = {"musdb16k": "musdb16k", "accompaniment": "accompaniment", "musdb_lyrics": "musdb_lyrics", "jamendolyrics": "jamendolyrics"}
csd = sorted(glob.glob(f"{DATA}/CSD/**/english", recursive=True))
if ok:
    for name, rel in parts.items():
        if os.path.exists(f"{DATA}/{rel}"):
            with tarfile.open(f"{PACK}/{name}.tar", "w") as t:
                t.add(f"{DATA}/{rel}", arcname=rel, filter=lambda ti: None if "/.git" in ti.name else ti)
    if csd:
        with tarfile.open(f"{PACK}/csd.tar", "w") as t:
            t.add(csd[0], arcname=os.path.relpath(csd[0], DATA))
    json.dump({"songs": len(glob.glob(f"{DATA}/musdb16k/*/*/vocals.wav"))}, open(f"{PACK}/info.json", "w"))
    shutil.rmtree(DATA)
    print(subprocess.run(f"du -sh {PACK}/*", shell=True, capture_output=True, text=True).stdout)
    print("READY - attach this notebook's Output to the training notebook")
else:
    print("NOT READY: MUSDB18 is incomplete. Run this notebook again with its own Output attached; it continues.")